# Notebook 08 — Multimodal Feature Integration and Preprocessing

**Project:** Machine Learning-Based Prediction of Parkinson’s Disease Progression Using PPMI  
**Stage:** Feature integration and leakage-safe preprocessing  
**Primary outcome:** Rapid motor progression at V06 (`rapid_progression_q75`)  
**New data source:** DaTSCAN Quant SBR screening-time tabular features (`EVENT_ID == SC`)  

---

## Objective

This notebook integrates baseline clinical predictors from Notebook 04 with screening-time DaTSCAN Quant SBR predictors from Notebook 07b. It preserves the same train/test split created in Notebook 04 and fits preprocessing steps on the training set only.

The notebook creates:

1. Clinical-only reference metadata from Notebook 04.
2. Clinical + DaTSCAN raw train/test splits.
3. Clinical + DaTSCAN processed train/test matrices.
4. Feature dictionaries and missingness summaries.
5. Quality-control outputs for the next modeling notebook.

**No machine learning model is trained in this notebook.**

## Scientific Background

The clinical-only model showed limited discrimination for rapid motor progression. DaTSCAN SBR features are biologically relevant because dopamine transporter signal reflects nigrostriatal dopaminergic integrity, which is directly related to Parkinsonian motor dysfunction. In PPMI, DaTSCAN SBR values are primarily available at screening (`SC`) rather than baseline (`BL`), so this notebook treats screening-time SBR as pre-baseline/baseline-available predictor information.

To prevent data leakage, this notebook:

- uses the same train/test split from Notebook 04;
- merges DaTSCAN features by `PATNO` only;
- uses only predictors available before or at baseline/screening;
- fits imputers, encoders, and scalers on training data only;
- applies the fitted preprocessing pipeline to the test set without refitting.

## Dataset Verification

Required input files:

From Notebook 04:

- `04_train_raw_split_before_preprocessing.csv`
- `05_test_raw_split_before_preprocessing.csv`
- `03_feature_type_dictionary.csv`

From Notebook 07b:

- `03_datscan_sbr_screening_feature_matrix.csv`
- `05_recommended_datscan_sbr_features_missing_le_30pct.csv`

Expected output folder:

`MyDrive/PPMI_PD_Progression/outputs/notebook_08_multimodal_preprocessing/`

In [ ]:
# ============================================================
# 01. Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# ============================================================
# 02. Imports and project paths
# ============================================================

from pathlib import Path
import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

import joblib

# ----------------------------
# Reproducibility settings
# ----------------------------

RANDOM_STATE = 42
TARGET_COL = "rapid_progression_q75"
ID_COL = "PATNO"

# ----------------------------
# Project folders
# ----------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/PPMI_PD_Progression")

NB04_DIR = PROJECT_DIR / "outputs" / "notebook_04_preprocessing"
NB07B_DIR = PROJECT_DIR / "outputs" / "notebook_07b_datscan_screening_inventory"
OUT_DIR = PROJECT_DIR / "outputs" / "notebook_08_multimodal_preprocessing"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("Notebook 04 output directory:", NB04_DIR, "| exists:", NB04_DIR.exists())
print("Notebook 07b output directory:", NB07B_DIR, "| exists:", NB07B_DIR.exists())
print("Notebook 08 output directory:", OUT_DIR, "| exists:", OUT_DIR.exists())

if not NB04_DIR.exists():
    raise FileNotFoundError(f"Notebook 04 directory not found: {NB04_DIR}")

if not NB07B_DIR.exists():
    raise FileNotFoundError(f"Notebook 07b directory not found: {NB07B_DIR}")

In [ ]:
# ============================================================
# 03. Verify required input files
# ============================================================

required_files = {
    "nb04_train_raw": NB04_DIR / "04_train_raw_split_before_preprocessing.csv",
    "nb04_test_raw": NB04_DIR / "05_test_raw_split_before_preprocessing.csv",
    "nb04_feature_types": NB04_DIR / "03_feature_type_dictionary.csv",
    "nb07b_datscan_matrix": NB07B_DIR / "03_datscan_sbr_screening_feature_matrix.csv",
    "nb07b_recommended_features": NB07B_DIR / "05_recommended_datscan_sbr_features_missing_le_30pct.csv",
}

file_check = []
for label, path in required_files.items():
    file_check.append({
        "input_label": label,
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else np.nan,
    })

file_check_df = pd.DataFrame(file_check)
display(file_check_df)
file_check_df.to_csv(OUT_DIR / "01_input_file_check.csv", index=False)

missing = file_check_df.loc[~file_check_df["exists"], "path"].tolist()
if missing:
    raise FileNotFoundError("Missing required input files:\n" + "\n".join(missing))

In [ ]:
# ============================================================
# 04. Load Notebook 04 clinical train/test raw splits
# ============================================================

train_raw = pd.read_csv(required_files["nb04_train_raw"], low_memory=False)
test_raw = pd.read_csv(required_files["nb04_test_raw"], low_memory=False)
feature_types = pd.read_csv(required_files["nb04_feature_types"], low_memory=False)

for df_name, df in [("train_raw", train_raw), ("test_raw", test_raw)]:
    if ID_COL not in df.columns:
        raise ValueError(f"{ID_COL} not found in {df_name}.")
    if TARGET_COL not in df.columns:
        raise ValueError(f"{TARGET_COL} not found in {df_name}.")
    df[ID_COL] = df[ID_COL].astype(str)

print("Clinical train raw shape:", train_raw.shape)
print("Clinical test raw shape:", test_raw.shape)
print("Feature type table shape:", feature_types.shape)

display(train_raw.head())
display(feature_types.head())

In [ ]:
# ============================================================
# 05. Load DaTSCAN SBR feature matrix and recommended feature list
# ============================================================

datscan = pd.read_csv(required_files["nb07b_datscan_matrix"], low_memory=False)
recommended = pd.read_csv(required_files["nb07b_recommended_features"], low_memory=False)

if ID_COL not in datscan.columns:
    raise ValueError(f"{ID_COL} not found in DaTSCAN feature matrix.")

datscan[ID_COL] = datscan[ID_COL].astype(str)

if "feature" not in recommended.columns:
    raise ValueError("Expected column 'feature' not found in recommended DaTSCAN feature list.")

recommended_datscan_features = [
    f for f in recommended["feature"].dropna().astype(str).tolist()
    if f in datscan.columns
]

if len(recommended_datscan_features) == 0:
    raise ValueError("No recommended DaTSCAN features were found in the DaTSCAN matrix.")

datscan_subset = datscan[[ID_COL] + recommended_datscan_features].copy()

# Confirm one row per participant.
duplicate_patno = int(datscan_subset[ID_COL].duplicated().sum())
if duplicate_patno > 0:
    datscan_subset = datscan_subset.drop_duplicates(subset=[ID_COL], keep="first")

print("DaTSCAN matrix shape:", datscan.shape)
print("Recommended DaTSCAN features:", len(recommended_datscan_features))
print("Duplicate PATNO rows removed:", duplicate_patno)

display(datscan_subset.head())

In [ ]:
# ============================================================
# 06. Merge clinical train/test splits with DaTSCAN features
# ============================================================

train_ids = set(train_raw[ID_COL].astype(str))
test_ids = set(test_raw[ID_COL].astype(str))
datscan_ids = set(datscan_subset[ID_COL].astype(str))

clinical_train_datscan_overlap = len(train_ids.intersection(datscan_ids))
clinical_test_datscan_overlap = len(test_ids.intersection(datscan_ids))

overlap_summary = pd.DataFrame({
    "set": ["train", "test", "overall"],
    "n_clinical_participants": [len(train_ids), len(test_ids), len(train_ids.union(test_ids))],
    "n_with_datscan": [
        clinical_train_datscan_overlap,
        clinical_test_datscan_overlap,
        len(train_ids.union(test_ids).intersection(datscan_ids)),
    ],
})
overlap_summary["pct_with_datscan"] = 100 * overlap_summary["n_with_datscan"] / overlap_summary["n_clinical_participants"]

display(overlap_summary)
overlap_summary.to_csv(OUT_DIR / "02_multimodal_overlap_summary.csv", index=False)

train_mm_raw = train_raw.merge(datscan_subset, on=ID_COL, how="left", validate="one_to_one")
test_mm_raw = test_raw.merge(datscan_subset, on=ID_COL, how="left", validate="one_to_one")

print("Train multimodal raw shape:", train_mm_raw.shape)
print("Test multimodal raw shape:", test_mm_raw.shape)

In [ ]:
# ============================================================
# 07. DaTSCAN missingness by split
# ============================================================

missing_rows = []
for split_name, split_df in [("train", train_mm_raw), ("test", test_mm_raw), ("combined", pd.concat([train_mm_raw, test_mm_raw], axis=0))]:
    for c in recommended_datscan_features:
        missing_rows.append({
            "set": split_name,
            "feature": c,
            "missing_n": int(split_df[c].isna().sum()),
            "missing_pct": round(100 * split_df[c].isna().mean(), 2),
            "non_missing_n": int(split_df[c].notna().sum()),
        })

split_missingness = pd.DataFrame(missing_rows)
split_missingness.to_csv(OUT_DIR / "03_datscan_feature_missingness_by_split.csv", index=False)

display(split_missingness.head(20))
print("Max training missingness among DaTSCAN features:", split_missingness.query("set == 'train'")["missing_pct"].max())
print("Max test missingness among DaTSCAN features:", split_missingness.query("set == 'test'")["missing_pct"].max())

In [ ]:
# ============================================================
# 08. Define multimodal feature types
# ============================================================

# Clinical feature types come from Notebook 04.
feature_types = feature_types.copy()
feature_types["predictor"] = feature_types["predictor"].astype(str)
feature_types["feature_type"] = feature_types["feature_type"].astype(str)

clinical_predictors = [
    c for c in train_raw.columns
    if c not in [ID_COL, TARGET_COL]
]

# Keep only predictors actually present in the current raw split.
feature_types = feature_types[feature_types["predictor"].isin(clinical_predictors)].copy()

# DaTSCAN SBR features are continuous numeric predictors.
datscan_feature_types = pd.DataFrame({
    "predictor": recommended_datscan_features,
    "feature_type": "continuous",
    "source": "DaTSCAN_SBR_SC",
})

clinical_feature_types = feature_types.copy()
clinical_feature_types["source"] = "clinical_baseline"

multimodal_feature_types = pd.concat([clinical_feature_types, datscan_feature_types], ignore_index=True)

# Handle any clinical columns not present in the Notebook 04 type dictionary.
typed_predictors = set(multimodal_feature_types["predictor"])
untyped_clinical = [c for c in clinical_predictors if c not in typed_predictors]

if untyped_clinical:
    extra_types = pd.DataFrame({
        "predictor": untyped_clinical,
        "feature_type": "continuous",
        "source": "clinical_baseline_untyped_default_continuous",
    })
    multimodal_feature_types = pd.concat([multimodal_feature_types, extra_types], ignore_index=True)

multimodal_feature_types.to_csv(OUT_DIR / "04_multimodal_feature_type_dictionary.csv", index=False)
display(multimodal_feature_types)

print("Clinical predictors:", len(clinical_predictors))
print("DaTSCAN predictors:", len(recommended_datscan_features))
print("Total raw predictors:", len(multimodal_feature_types))

In [ ]:
# ============================================================
# 09. Prepare X/y while preserving Notebook 04 train/test split
# ============================================================

predictor_cols = multimodal_feature_types["predictor"].tolist()

# Safety: keep only columns present in both train and test matrices.
predictor_cols = [c for c in predictor_cols if c in train_mm_raw.columns and c in test_mm_raw.columns]

X_train_raw = train_mm_raw[predictor_cols].copy()
X_test_raw = test_mm_raw[predictor_cols].copy()

y_train = train_mm_raw[TARGET_COL].astype(int).copy()
y_test = test_mm_raw[TARGET_COL].astype(int).copy()
ids_train = train_mm_raw[ID_COL].astype(str).copy()
ids_test = test_mm_raw[ID_COL].astype(str).copy()

# Remove constant predictors using training data only.
train_unique_counts = X_train_raw.nunique(dropna=True)
constant_cols = train_unique_counts[train_unique_counts <= 1].index.tolist()

X_train_raw_reduced = X_train_raw.drop(columns=constant_cols)
X_test_raw_reduced = X_test_raw.drop(columns=constant_cols)

if constant_cols:
    print("Constant predictors dropped from multimodal matrix:")
    for c in constant_cols:
        print("-", c)
else:
    print("No constant predictors detected in training set.")

pd.DataFrame({"dropped_constant_predictor": constant_cols}).to_csv(
    OUT_DIR / "05_dropped_constant_predictors_multimodal.csv", index=False
)

print("X_train raw reduced shape:", X_train_raw_reduced.shape)
print("X_test raw reduced shape:", X_test_raw_reduced.shape)

In [ ]:
# ============================================================
# 10. Build leakage-safe preprocessing pipeline
# ============================================================

active_feature_types = multimodal_feature_types[
    multimodal_feature_types["predictor"].isin(X_train_raw_reduced.columns)
].copy()

continuous_features = active_feature_types.loc[active_feature_types["feature_type"] == "continuous", "predictor"].tolist()
binary_features = active_feature_types.loc[active_feature_types["feature_type"] == "binary", "predictor"].tolist()
categorical_features = active_feature_types.loc[active_feature_types["feature_type"] == "categorical", "predictor"].tolist()

# Any unassigned columns are treated as continuous by default.
assigned = set(continuous_features + binary_features + categorical_features)
unassigned = [c for c in X_train_raw_reduced.columns if c not in assigned]
if unassigned:
    print("Unassigned columns treated as continuous:")
    for c in unassigned:
        print("-", c)
    continuous_features += unassigned

# Robust OneHotEncoder for different scikit-learn versions.
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

continuous_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

binary_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("continuous", continuous_pipeline, continuous_features),
        ("binary", binary_pipeline, binary_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

X_train_processed = preprocessor.fit_transform(X_train_raw_reduced)
X_test_processed = preprocessor.transform(X_test_raw_reduced)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

In [ ]:
# ============================================================
# 11. Recover processed feature names
# ============================================================

try:
    processed_feature_names = preprocessor.get_feature_names_out().tolist()
except Exception:
    processed_feature_names = []
    processed_feature_names += [f"continuous__{c}" for c in continuous_features]
    processed_feature_names += [f"binary__{c}" for c in binary_features]

    if categorical_features:
        cat_transformer = preprocessor.named_transformers_["categorical"]
        onehot = cat_transformer.named_steps["onehot"]
        try:
            cat_names = onehot.get_feature_names_out(categorical_features).tolist()
        except Exception:
            cat_names = [f"{c}__category" for c in categorical_features]
        processed_feature_names += [f"categorical__{c}" for c in cat_names]

X_train_processed_df = pd.DataFrame(X_train_processed, columns=processed_feature_names)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=processed_feature_names)

processed_names_df = pd.DataFrame({"processed_feature_name": processed_feature_names})
processed_names_df.to_csv(OUT_DIR / "09_multimodal_processed_feature_names.csv", index=False)

display(processed_names_df.head(20))
print("Processed feature count:", len(processed_feature_names))
print("Missing values after processing — train:", int(X_train_processed_df.isna().sum().sum()))
print("Missing values after processing — test:", int(X_test_processed_df.isna().sum().sum()))

In [ ]:
# ============================================================
# 12. Save multimodal raw and processed matrices
# ============================================================

train_mm_raw_out = pd.concat([
    ids_train.reset_index(drop=True).rename(ID_COL),
    y_train.reset_index(drop=True).rename(TARGET_COL),
    X_train_raw_reduced.reset_index(drop=True),
], axis=1)

test_mm_raw_out = pd.concat([
    ids_test.reset_index(drop=True).rename(ID_COL),
    y_test.reset_index(drop=True).rename(TARGET_COL),
    X_test_raw_reduced.reset_index(drop=True),
], axis=1)

train_mm_processed_out = pd.concat([
    ids_train.reset_index(drop=True).rename(ID_COL),
    y_train.reset_index(drop=True).rename(TARGET_COL),
    X_train_processed_df.reset_index(drop=True),
], axis=1)

test_mm_processed_out = pd.concat([
    ids_test.reset_index(drop=True).rename(ID_COL),
    y_test.reset_index(drop=True).rename(TARGET_COL),
    X_test_processed_df.reset_index(drop=True),
], axis=1)

train_mm_raw_out.to_csv(OUT_DIR / "06_multimodal_train_raw_split_before_preprocessing.csv", index=False)
test_mm_raw_out.to_csv(OUT_DIR / "07_multimodal_test_raw_split_before_preprocessing.csv", index=False)
train_mm_processed_out.to_csv(OUT_DIR / "08_multimodal_train_processed_matrix.csv", index=False)
test_mm_processed_out.to_csv(OUT_DIR / "09_multimodal_test_processed_matrix.csv", index=False)

joblib.dump(preprocessor, OUT_DIR / "10_fitted_multimodal_preprocessing_pipeline.joblib")

print("Saved multimodal matrices and preprocessing pipeline to:", OUT_DIR)

In [ ]:
# ============================================================
# 13. Feature set manifest and sensitivity feature list
# ============================================================

clinical_only_processed_names_path = NB04_DIR / "08_processed_feature_names.csv"
if clinical_only_processed_names_path.exists():
    clinical_only_processed_names = pd.read_csv(clinical_only_processed_names_path)["processed_feature_name"].tolist()
else:
    clinical_only_processed_names = []

manifest = pd.DataFrame([
    {
        "feature_set": "clinical_only_reference_from_notebook_04",
        "raw_predictor_count": len(clinical_predictors),
        "processed_feature_count": len(clinical_only_processed_names),
        "train_n": len(train_raw),
        "test_n": len(test_raw),
        "source_output_folder": str(NB04_DIR),
    },
    {
        "feature_set": "clinical_plus_datscan_sbr_sc",
        "raw_predictor_count": len(X_train_raw_reduced.columns),
        "processed_feature_count": len(processed_feature_names),
        "train_n": len(train_mm_processed_out),
        "test_n": len(test_mm_processed_out),
        "source_output_folder": str(OUT_DIR),
    },
])

manifest.to_csv(OUT_DIR / "11_feature_set_manifest.csv", index=False)
display(manifest)

# Sensitivity analysis without baseline_NP3TOT.
processed_feature_names_without_baseline_np3tot = [
    c for c in processed_feature_names
    if "baseline_NP3TOT" not in c
]

pd.DataFrame({"processed_feature_name": processed_feature_names_without_baseline_np3tot}).to_csv(
    OUT_DIR / "12_multimodal_processed_feature_names_without_baseline_NP3TOT.csv",
    index=False,
)

sensitivity_plan = pd.DataFrame([
    {
        "sensitivity_analysis": "multimodal_exclude_baseline_NP3TOT",
        "reason": "Outcome is based on change in NP3TOT; remove baseline_NP3TOT to assess robustness.",
        "processed_feature_count_primary": len(processed_feature_names),
        "processed_feature_count_sensitivity": len(processed_feature_names_without_baseline_np3tot),
    }
])

sensitivity_plan.to_csv(OUT_DIR / "13_multimodal_sensitivity_feature_set_plan.csv", index=False)
display(sensitivity_plan)

In [ ]:
# ============================================================
# 14. Quality Control Checklist
# ============================================================

qc_rows = []

def add_qc(item, status, details):
    qc_rows.append({"qc_item": item, "status": status, "details": details})

add_qc(
    "Notebook 04 train/test raw splits loaded",
    "PASS" if train_raw.shape[0] > 0 and test_raw.shape[0] > 0 else "FAIL",
    f"train={train_raw.shape}; test={test_raw.shape}"
)

add_qc(
    "Notebook 07b DaTSCAN SBR matrix loaded",
    "PASS" if datscan_subset.shape[0] > 0 and len(recommended_datscan_features) > 0 else "FAIL",
    f"rows={datscan_subset.shape[0]}; recommended_features={len(recommended_datscan_features)}"
)

add_qc(
    "Train/test split preserved from Notebook 04",
    "PASS" if len(ids_train) == train_raw.shape[0] and len(ids_test) == test_raw.shape[0] else "FAIL",
    f"train_n={len(ids_train)}; test_n={len(ids_test)}"
)

add_qc(
    "Participant IDs remain unique within train and test",
    "PASS" if (not ids_train.duplicated().any() and not ids_test.duplicated().any()) else "FAIL",
    f"train_duplicates={int(ids_train.duplicated().sum())}; test_duplicates={int(ids_test.duplicated().sum())}"
)

add_qc(
    "No overlap between train and test participant IDs",
    "PASS" if len(set(ids_train).intersection(set(ids_test))) == 0 else "FAIL",
    f"overlap_n={len(set(ids_train).intersection(set(ids_test)))}"
)

add_qc(
    "DaTSCAN overlap adequate",
    "PASS" if overlap_summary.loc[overlap_summary["set"] == "overall", "pct_with_datscan"].iloc[0] >= 70 else "CHECK",
    f"overall_datscan_coverage_pct={overlap_summary.loc[overlap_summary['set'] == 'overall', 'pct_with_datscan'].iloc[0]:.2f}"
)

add_qc(
    "Preprocessor fit on training data only",
    "PASS",
    "fit_transform used for training set; transform used for test set."
)

add_qc(
    "No missing values after preprocessing",
    "PASS" if (X_train_processed_df.isna().sum().sum() == 0 and X_test_processed_df.isna().sum().sum() == 0) else "FAIL",
    f"train_missing={int(X_train_processed_df.isna().sum().sum())}; test_missing={int(X_test_processed_df.isna().sum().sum())}"
)

add_qc(
    "Target distribution preserved",
    "PASS" if abs(y_train.mean() - y_test.mean()) < 0.05 else "CHECK",
    f"train_positive_pct={100*y_train.mean():.2f}; test_positive_pct={100*y_test.mean():.2f}"
)

add_qc(
    "No ML modeling performed",
    "PASS",
    "This notebook performs feature integration and preprocessing only."
)

qc = pd.DataFrame(qc_rows)
qc.to_csv(OUT_DIR / "14_quality_control_checklist.csv", index=False)
display(qc)

if (qc["status"] == "FAIL").any():
    raise RuntimeError("One or more QC checks failed. Review 14_quality_control_checklist.csv before proceeding.")

In [ ]:
# ============================================================
# 15. Summary report
# ============================================================

summary = f"""
Notebook 08 — Multimodal Feature Integration and Preprocessing
================================================================

Input folders:
- Notebook 04: {NB04_DIR}
- Notebook 07b: {NB07B_DIR}

Input clinical split:
- Train raw shape: {train_raw.shape}
- Test raw shape: {test_raw.shape}

DaTSCAN SBR input:
- DaTSCAN feature matrix shape: {datscan.shape}
- Recommended DaTSCAN SBR features used: {len(recommended_datscan_features)}

DaTSCAN overlap:
{overlap_summary.to_string(index=False)}

Multimodal raw predictor matrix:
- Train raw predictors after constant removal: {X_train_raw_reduced.shape}
- Test raw predictors after constant removal: {X_test_raw_reduced.shape}
- Dropped constant predictors: {constant_cols}

Feature types:
- Continuous: {len(continuous_features)}
- Binary: {len(binary_features)}
- Categorical: {len(categorical_features)}

Processed multimodal matrix:
- Train processed shape: {X_train_processed_df.shape}
- Test processed shape: {X_test_processed_df.shape}
- Missing values after preprocessing: train={int(X_train_processed_df.isna().sum().sum())}, test={int(X_test_processed_df.isna().sum().sum())}

Output files:
- 01_input_file_check.csv
- 02_multimodal_overlap_summary.csv
- 03_datscan_feature_missingness_by_split.csv
- 04_multimodal_feature_type_dictionary.csv
- 05_dropped_constant_predictors_multimodal.csv
- 06_multimodal_train_raw_split_before_preprocessing.csv
- 07_multimodal_test_raw_split_before_preprocessing.csv
- 08_multimodal_train_processed_matrix.csv
- 09_multimodal_test_processed_matrix.csv
- 10_fitted_multimodal_preprocessing_pipeline.joblib
- 11_feature_set_manifest.csv
- 12_multimodal_processed_feature_names_without_baseline_NP3TOT.csv
- 13_multimodal_sensitivity_feature_set_plan.csv
- 14_quality_control_checklist.csv
- 15_notebook_08_summary_report.txt

Decision:
The multimodal clinical + DaTSCAN SBR feature set is prepared for Notebook 09 modeling.
No machine learning model was trained in this notebook.

Output folder:
{OUT_DIR}
"""

print(summary)

with open(OUT_DIR / "15_notebook_08_summary_report.txt", "w", encoding="utf-8") as f:
    f.write(summary)

## Scientific Interpretation

After this notebook is run, the key decision is whether the DaTSCAN SBR features provide adequate coverage and produce clean processed matrices. If QC passes, the next notebook can compare:

1. Clinical-only model performance.
2. Clinical + DaTSCAN SBR model performance.
3. Sensitivity analysis excluding `baseline_NP3TOT`.

A clinically meaningful improvement would require better discrimination and/or improved sensitivity for rapid progressors, not only a small increase in ROC-AUC.

## Quality Control Checklist

The notebook automatically checks:

- Notebook 04 outputs exist.
- Notebook 07b outputs exist.
- Train/test split is preserved.
- Participant IDs are unique.
- Train/test participant IDs do not overlap.
- DaTSCAN coverage is adequate.
- Preprocessing is fitted only on training data.
- No missing values remain after preprocessing.
- No machine learning modeling is performed.

## Expected Output

After running this notebook, upload the following files for review:

- `02_multimodal_overlap_summary.csv`
- `03_datscan_feature_missingness_by_split.csv`
- `04_multimodal_feature_type_dictionary.csv`
- `11_feature_set_manifest.csv`
- `13_multimodal_sensitivity_feature_set_plan.csv`
- `14_quality_control_checklist.csv`
- `15_notebook_08_summary_report.txt`

Compress them as:

`Notebook_08_outputs_review.zip`